In [4]:
import pandas as pd

# ================== 工具函数 ==================
# 颜色渲染：正数红色，其余默认
def colorize(val: float, width=10):
    if pd.isna(val):
        return " " * width
    s = f"{val:.3f}".rjust(width)  # 固定宽度，保证对齐
    if val > 0:
        return f"\033[91m{s}\033[0m"  # 红色
    return s

# 表格打印（对齐 + 颜色）
def print_colored_table(df: pd.DataFrame, title: str):
    print(title)
    # 打印列名
    header = " " * 10 + "".join(c.rjust(10) for c in df.columns)
    print(header)
    # 打印每行
    for idx, row in df.iterrows():
        row_str = str(idx).ljust(10)
        for val in row:
            row_str += colorize(val, width=10)
        print(row_str)
    print()


# ================== 主逻辑 ==================
# 读取结果
df = pd.read_csv("./results/result_summary.csv")

loss_list = [f"loss{i}" for i in range(1, 6)]
topk_list = ["Top 3", "Top 5", "Top 10", "Top 20"]

# ========== 生成表 (平均提升百分比) ==========
def make_tables(df, label):
    for topk in topk_list:
        df_topk = df[df["TopK"] == topk]
        pivot = (
            df_topk[df_topk["Loss"].isin(loss_list)]
            .groupby(["Model", "Loss"])["RelDiff(%)"]
            .mean()
            .reset_index()
        )
        table = pivot.pivot(index="Loss", columns="Model", values="RelDiff(%)").round(3)
        print_colored_table(table, f"\n=== {label} | {topk} ===")


print("========== 汇总表 ==========")
make_tables(df, "Results")


========== 汇总表 ==========

=== Results | Top 3 ===
                 NCL       SGL    SimGCL   XSimGCL
loss1         -4.357    -4.393     0.316    -0.343
loss2         -0.416    -2.129     3.545     2.690
loss3          0.371    -1.607     0.503     2.531
loss4         -3.724    -7.513    -1.568    -2.414
loss5         -0.293    -0.834     0.226     3.113


=== Results | Top 5 ===
                 NCL       SGL    SimGCL   XSimGCL
loss1         -4.076    -3.535    -0.516    -2.444
loss2         -1.100    -1.847     2.507    -0.122
loss3         -0.484    -1.144     0.279     0.159
loss4         -4.014    -6.401    -2.165    -2.187
loss5         -0.239     0.545     0.825    -0.080


=== Results | Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
loss1         -3.298    -3.089    -0.870    -1.360
loss2         -0.336    -1.542     1.467     0.215
loss3          0.010    -0.838    -0.218    -0.089
loss4         -3.254    -4.683    -1.588    -1.034
loss5         -0.165     0.35

In [10]:
import pandas as pd
from colorama import Fore, Style

for model, subdf in df.groupby("Model"):
    print(f"\n===== Model: {model} =====")

    # 先转成字符串表格
    table_str = subdf.to_string(index=False)

    # 按行拆分
    lines = table_str.split("\n")

    # 第一行是表头，原样打印
    print(lines[0])

    # 从第二行开始逐行处理
    for i, (_, row) in enumerate(subdf.iterrows(), start=1):
        line = lines[i]
        if row["RelDiff(%)"] > 0:
            print(Fore.RED + line + Style.RESET_ALL)
        else:
            print(line)



===== Model: NCL =====
Model    Campus  Loss   TopK    Metric  Baseline(loss0)   Value  AbsDiff  RelDiff(%)
  NCL campus_15 loss1 Top 20 Hit Ratio          0.51416 0.50363 -0.01053   -2.048001
  NCL campus_15 loss1 Top 20 Precision          0.12456 0.12201 -0.00255   -2.047206
  NCL campus_15 loss1 Top 20    Recall          0.54863 0.54769 -0.00094   -0.171336
  NCL campus_15 loss1 Top 20      NDCG          0.40486 0.39633 -0.00853   -2.106901
  NCL campus_15 loss2 Top 20 Hit Ratio          0.51416 0.51408 -0.00008   -0.015559
  NCL campus_15 loss2 Top 20 Precision          0.12456 0.12454 -0.00002   -0.016057
  NCL campus_15 loss2 Top 20    Recall          0.54863 0.55072  0.00209    0.380949
  NCL campus_15 loss2 Top 20      NDCG          0.40486 0.40274 -0.00212   -0.523638
  NCL campus_15 loss3 Top 20 Hit Ratio          0.51416 0.51871  0.00455    0.884939
  NCL campus_15 loss3 Top 20 Precision          0.12456 0.12567  0.00111    0.891137
  NCL campus_15 loss3 Top 20    Recall   